In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### Load the Data

In [1]:
import pandas as pd

file_path = "/kaggle/input/transactions/synthetic_fraud_data.csv"
data = pd.read_csv(file_path)

# Preview the data
print(data.head())
print(data.columns)

  transaction_id customer_id       card_number  \
0    TX_a0ad2a2a  CUST_72886  6646734767813109   
1    TX_3599c101  CUST_70474   376800864692727   
2    TX_a9461c6d  CUST_10715  5251909460951913   
3    TX_7be21fc4  CUST_16193   376079286931183   
4    TX_150f490b  CUST_87572  6172948052178810   

                          timestamp merchant_category merchant_type  \
0  2024-09-30 00:00:01.034820+00:00        Restaurant     fast_food   
1  2024-09-30 00:00:01.764464+00:00     Entertainment        gaming   
2  2024-09-30 00:00:02.273762+00:00           Grocery      physical   
3  2024-09-30 00:00:02.297466+00:00               Gas         major   
4  2024-09-30 00:00:02.544063+00:00        Healthcare       medical   

         merchant     amount currency    country  ...   device channel  \
0       Taco Bell     294.87      GBP         UK  ...  iOS App  mobile   
1           Steam    3368.97      BRL     Brazil  ...     Edge     web   
2     Whole Foods  102582.38      JPY      Japan  

### Parse velocity_last_hour to Extract Features

In [2]:
import ast

# Parse the string dictionary
data['velocity_last_hour_parsed'] = data['velocity_last_hour'].apply(ast.literal_eval)

# Extract fields
data['velocity_num_transactions'] = data['velocity_last_hour_parsed'].apply(lambda x: x['num_transactions'])
data['velocity_total_amount'] = data['velocity_last_hour_parsed'].apply(lambda x: x['total_amount'])
data['velocity_unique_merchants'] = data['velocity_last_hour_parsed'].apply(lambda x: x['unique_merchants'])
data['velocity_unique_countries'] = data['velocity_last_hour_parsed'].apply(lambda x: x['unique_countries'])
data['velocity_max_single_amount'] = data['velocity_last_hour_parsed'].apply(lambda x: x['max_single_amount'])


### Drop Columns That Are IDs or Unnecessary

In [3]:
data.drop([
    'velocity_last_hour',
    'velocity_last_hour_parsed',
    'transaction_id',
    'customer_id',
    'card_number',
    'timestamp',
    'device_fingerprint',
    'ip_address',
    'merchant',
    'currency'
], axis=1, inplace=True)

### Standard Scale Numerical Columns

In [4]:
from sklearn.preprocessing import StandardScaler

numerical_cols = [
    'amount',
    'distance_from_home',
    'transaction_hour',
    'velocity_num_transactions',
    'velocity_total_amount',
    'velocity_unique_merchants',
    'velocity_unique_countries',
    'velocity_max_single_amount'
]

scaler = StandardScaler()
data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

###  Label Encode Limited-Categories Columns

In [5]:
from sklearn.preprocessing import LabelEncoder

categorical_cols_label = ['country', 'city_size', 'card_present']

le = LabelEncoder()
for col in categorical_cols_label:
    data[col] = le.fit_transform(data[col].astype(str))

### One-Hot Encode Remaining Categorical Columns

In [6]:
categorical_cols_onehot = [
    'merchant_category',
    'merchant_type',
    'city',
    'card_type',
    'device',
    'channel',
    'high_risk_merchant'
]

data = pd.get_dummies(data, columns=categorical_cols_onehot, drop_first=True)

### Split Data into Train/Test

In [7]:
from sklearn.model_selection import train_test_split

X = data.drop('is_fraud', axis=1)
y = data['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

### Train and Evaluate Machine Learning Algorithm

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"\n--- {name} ---")
    print(classification_report(y_test, y_pred))
    print(f"ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
    print(confusion_matrix(y_test, y_pred))


--- Logistic Regression ---
              precision    recall  f1-score   support

       False       0.94      0.97      0.95   1197810
        True       0.84      0.74      0.79    298944

    accuracy                           0.92   1496754
   macro avg       0.89      0.85      0.87   1496754
weighted avg       0.92      0.92      0.92   1496754

ROC AUC Score: 0.9570
[[1157002   40808]
 [  77141  221803]]
